# Class 4: Categorical Encoding, Transformers & Missing Values

In this notebook, we cover:
1. Creating & loading datasets with **missing values** in both numerical and categorical columns.
2. Identifying and inspecting null values (`isnull()`, `isna()`, percentages).
3. Imputing missing values using `SimpleImputer` (mean/median for numerical, mode for categorical).
4. Encoding categorical variables using `OneHotEncoder` and `OrdinalEncoder`.
5. Combining preprocessing steps seamlessly using Scikit-Learn's `ColumnTransformer` and `Pipeline`.

## 1. Import Libraries & Create Dataset with Missing Values

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Sample dataset containing both numerical and categorical features with null values
data = {
    "Name": ["Aarav", "Priya", "Rohit", "Sneha", "Arjun", "Neha", "Karan", "Pooja", "Vikram", "Ananya"],
    "Gender": ["Male", "Female", np.nan, "Female", "Male", "Female", "Male", np.nan, "Male", "Female"],           # Categorical (Nominal)
    "Education": ["Bachelor", "Master", "PhD", np.nan, "Master", "Bachelor", np.nan, "PhD", "Bachelor", "Master"], # Categorical (Ordinal)
    "City": ["Mumbai", "Delhi", "Bengaluru", "Mumbai", np.nan, "Pune", "Delhi", "Bengaluru", np.nan, "Pune"],     # Categorical (Nominal)
    "Age": [25, 28, np.nan, 22, 30, np.nan, 27, 24, 35, 29],                                                    # Numerical
    "Salary": [50000, 62000, 58000, np.nan, 75000, 48000, 67000, np.nan, 85000, 72000],                         # Numerical
    "Experience_Years": [2, 4, 3, 1, np.nan, 3, np.nan, 2, 8, 5]                                                # Numerical
}

df = pd.DataFrame(data)
df

## 2. Inspecting Missing Values

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Sum of missing values per column
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_pct
})
missing_summary

## 3. Handling Missing Values

We use `SimpleImputer`:
- **Numerical Columns**: Replace nulls with `mean` or `median`.
- **Categorical Columns**: Replace nulls with `most_frequent` (mode).

In [ ]:
from sklearn.impute import SimpleImputer

# Imputing numerical columns with median
num_imputer = SimpleImputer(strategy='median')
num_cols = ['Age', 'Salary', 'Experience_Years']
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Imputing categorical columns with mode (most frequent)
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_cols = ['Gender', 'Education', 'City']
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Verify no missing values remain
print("Remaining missing values:")
print(df.isnull().sum())
df

## 4. Categorical Encoding

### A. One-Hot Encoding (Nominal Features)
Used when categories have no natural order (e.g., `Gender`, `City`).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_nominal = ohe.fit_transform(df[['Gender', 'City']])
encoded_df = pd.DataFrame(encoded_nominal, columns=ohe.get_feature_names_out(['Gender', 'City']))
encoded_df

### B. Ordinal Encoding (Ordered Categorical Features)
Used when categories have inherent ranking (e.g., `Education`: Bachelor < Master < PhD).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

education_order = [['Bachelor', 'Master', 'PhD']]
ordinal_enc = OrdinalEncoder(categories=education_order)
df['Education_Encoded'] = ordinal_enc.fit_transform(df[['Education']])
df[['Education', 'Education_Encoded']]

## 5. End-to-End Pipeline with ColumnTransformer

In real-world machine learning, we combine **Imputation**, **Encoding**, and **Feature Scaling** into a single Scikit-Learn `ColumnTransformer` to avoid data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Re-create raw data with nulls to test pipeline from scratch
df_raw = pd.DataFrame(data)

numeric_features = ['Age', 'Salary', 'Experience_Years']
nominal_features = ['Gender', 'City']
ordinal_features = ['Education']

# 1. Numeric pipeline: Median Imputer + Standard Scaler
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Nominal pipeline: Most Frequent Imputer + OneHotEncoder
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Ordinal pipeline: Most Frequent Imputer + OrdinalEncoder
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Bachelor', 'Master', 'PhD']]))
])

# Combine all in ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

# Fit and transform the dataset
transformed_array = preprocessor.fit_transform(df_raw)

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()
transformed_df = pd.DataFrame(transformed_array, columns=feature_names)
transformed_df